In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
import os
import time
from datetime import datetime

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# بخش 1: تعریف توابع تحلیل (2 تابع مجزا)
# ============================================================================

def analysis_1_correlation_full(file_path, output_filename1, output_filename2):
    """
    تحلیل همبستگی کامل - کد شماره 1
    شامل: Rolling Correlation + خروجی Long + خروجی Wide
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 1 (همبستگی کامل - Long + Wide)")
    print(f"{'='*60}")
    
    all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                    'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    target_sensors = all_features
    
    try:
        # خواندن فایل
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ فایل ورودی با موفقیت خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return False
    
    # پیش‌پردازش و حذف داده‌های پرت
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
    
    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
    
    before_count = len(df_raw)
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    # بازه ۳۰ روز آخر
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    print(f"📅 شروع تحلیل Rolling از تاریخ: {start_analysis_date}")
    
    # محاسبه Rolling Correlation
    print("🔄 مرحله 2: محاسبه ماتریس کوریلیشن متحرک...")
    
    results_list = []
    
    for target in target_sensors:
        temp_storage = {}
        
        for feature in all_features:
            if target == feature:
                temp_storage[feature] = np.nan
                continue
            
            rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
            temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]
        
        target_df_wide = pd.DataFrame(temp_storage)
        target_df_wide['AssetID'] = target
        target_df_wide = target_df_wide.reset_index()
        cols = ['date', 'AssetID'] + all_features
        results_list.append(target_df_wide[cols])
    
    df_output2 = pd.concat(results_list, ignore_index=True)
    print(f"   ✅ خروجی همبستگی (Wide): {len(df_output2):,} رکورد")
    
    # تولید خروجی Long از روی Wide
    df_output1 = df_output2.melt(id_vars=['date', 'AssetID'], 
                                 value_vars=all_features, 
                                 var_name='AssetID_correlation', 
                                 value_name='correlation_value')
    df_output1 = df_output1.dropna(subset=['correlation_value'])
    print(f"   ✅ خروجی همبستگی (Long): {len(df_output1):,} رکورد")
    
    # ذخیره خروجی‌ها
    print("💾 مرحله 3: ذخیره خروجی‌ها...")
    
    try:
        os.makedirs(os.path.dirname(output_filename1), exist_ok=True)
        df_output1.to_excel(output_filename1, index=False)
        df_output2.to_excel(output_filename2, index=False)
        
        print(f"\n✅ عملیات با موفقیت پایان یافت.")
        print(f"📂 فایل ۱ (ساختار ردیفی - Long): {output_filename1}")
        print(f"📂 فایل ۲ (ساختار ستونی - Wide): {output_filename2}")
        print(f"\n📊 آمار نهایی:")
        print(f"   فایل ۱: {len(df_output1):,} رکورد")
        print(f"   فایل ۲: {len(df_output2):,} رکورد")
        return True
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل‌ها: {e}")
        return False


def analysis_2_correlation_wide_only(file_path, output_filename2):
    """
    تحلیل همبستگی - فقط خروجی Wide - کد شماره 2
    شامل: Rolling Correlation + فقط خروجی Wide
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 2 (همبستگی - فقط Wide)")
    print(f"{'='*60}")
    
    all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                    'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    target_sensors = all_features
    
    try:
        # خواندن فایل
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ فایل ورودی با موفقیت خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return False
    
    # پیش‌پردازش و حذف داده‌های پرت
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
    
    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
    
    before_count = len(df_raw)
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    # بازه ۳۰ روز آخر
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    print(f"📅 شروع تحلیل Rolling از تاریخ: {start_analysis_date}")
    
    # محاسبه Rolling Correlation
    print("🔄 مرحله 2: محاسبه ماتریس کوریلیشن متحرک...")
    
    results_list = []
    
    for target in target_sensors:
        temp_storage = {}
        
        for feature in all_features:
            if target == feature:
                temp_storage[feature] = np.nan
                continue
            
            rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
            temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]
        
        target_df_wide = pd.DataFrame(temp_storage)
        target_df_wide['AssetID'] = target
        target_df_wide = target_df_wide.reset_index()
        cols = ['date', 'AssetID'] + all_features
        results_list.append(target_df_wide[cols])
    
    df_output2 = pd.concat(results_list, ignore_index=True)
    print(f"   ✅ خروجی همبستگی (Wide): {len(df_output2):,} رکورد")
    
    # ذخیره خروجی
    print("💾 مرحله 3: ذخیره خروجی...")
    
    try:
        os.makedirs(os.path.dirname(output_filename2), exist_ok=True)
        df_output2.to_excel(output_filename2, index=False)
        
        print(f"\n✅ عملیات با موفقیت پایان یافت.")
        print(f"📂 فایل (ساختار ستونی - Wide): {output_filename2}")
        print(f"📊 تعداد رکوردهای نهایی: {len(df_output2):,}")
        print(f"📋 تعداد ستون‌ها: {len(df_output2.columns)}")
        return True
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل‌ها: {e}")
        return False


# ============================================================================
# بخش 2: تعریف وظایف (Jobs) - 2 وظیفه
# ============================================================================

def get_analysis_jobs():
    """
    تعریف ۲ وظیفه تحلیل با مسیرهای ورودی و خروجی مربوطه
    """
    base_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
    out_base = r'outputs\G11\dsas_g11_generator_bearings_correlation_detection\correlation_detection\dsas_g11_generator_bearings_correlation_detection_output'
    
    jobs = [
        {
            'name': 'Analysis 1 - Correlation Full (Long + Wide)',
            'function': analysis_1_correlation_full,
            'file_path': base_path,
            'output_filename1': f'{out_base}1.xlsx',
            'output_filename2': f'{out_base}2.xlsx'
        },
        {
            'name': 'Analysis 2 - Correlation Wide Only',
            'function': analysis_2_correlation_wide_only,
            'file_path': base_path,
            'output_filename2': f'{out_base}2.xlsx'
        }
    ]
    return jobs


def run_all_analyses():
    """
    اجرای تمام ۲ تحلیل به ترتیب
    """
    print("\n" + "="*80)
    print(f"🚀 شروع اجرای همه تحلیل‌ها در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    print("📋 لیست تحلیلها:")
    print("   1. تحلیل همبستگی کامل (Long + Wide)")
    print("   2. تحلیل همبستگی (فقط Wide)")
    print("="*80)
    
    jobs = get_analysis_jobs()
    results = []
    
    # وظیفه ۱: تحلیل کامل (Long + Wide)
    job1 = jobs[0]
    print(f"\n{'#'*80}")
    print(f"# اجرای وظیفه 1 از {len(jobs)}: {job1['name']}")
    print(f"{'#'*80}")
    
    try:
        success = job1['function'](job1['file_path'], job1['output_filename1'], job1['output_filename2'])
        results.append({
            'job_name': job1['name'],
            'success': success,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
        if success:
            print(f"✅ وظیفه 1 با موفقیت کامل شد")
        else:
            print(f"❌ وظیفه 1 با شکست مواجه شد")
    except Exception as e:
        print(f"❌ خطای غیرمنتظره در وظیفه 1: {e}")
        results.append({
            'job_name': job1['name'],
            'success': False,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'error': str(e)
        })
    
    # وظیفه ۲: تحلیل فقط Wide
    job2 = jobs[1]
    print(f"\n{'#'*80}")
    print(f"# اجرای وظیفه 2 از {len(jobs)}: {job2['name']}")
    print(f"{'#'*80}")
    
    try:
        success = job2['function'](job2['file_path'], job2['output_filename2'])
        results.append({
            'job_name': job2['name'],
            'success': success,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
        if success:
            print(f"✅ وظیفه 2 با موفقیت کامل شد")
        else:
            print(f"❌ وظیفه 2 با شکست مواجه شد")
    except Exception as e:
        print(f"❌ خطای غیرمنتظره در وظیفه 2: {e}")
        results.append({
            'job_name': job2['name'],
            'success': False,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'error': str(e)
        })
    
    # گزارش نهایی
    print("\n" + "="*80)
    print("📊 گزارش نهایی اجرای همه تحلیل‌ها")
    print("="*80)
    
    success_count = sum(1 for r in results if r['success'])
    total_count = len(results)
    
    print(f"✅ موفق: {success_count} از {total_count}")
    print(f"❌ ناموفق: {total_count - success_count} از {total_count}")
    
    for r in results:
        status = "✅" if r['success'] else "❌"
        print(f"   {status} {r['job_name']} - {r['timestamp']}")
    
    print("="*80)
    print(f"🏁 پایان اجرا در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    return results


# ============================================================================
# بخش 3: زمان‌بندی (Scheduler) با دو زمان ۹:۰۰ و ۲۱:۰۰
# ============================================================================

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز ساعت ۹:۰۰ و ۲۱:۰۰)
    """
    print("="*80)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل همبستگی ژنراتور")
    print("="*80)
    print("📋 شامل ۲ تحلیل:")
    print("   1. تحلیل همبستگی کامل (Long + Wide)")
    print("   2. تحلیل همبستگی (فقط Wide)")
    print("="*80)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 09:00")
    print("   - ساعت 21:00")
    print("="*80)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*80)
    
    last_run_times = {}  # ذخیره زمان‌های اجرا شده
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص (۹:۰۰ و ۲۱:۰۰)
            if current_time in ["17:41", "17:43"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_times.get(current_time) != now.strftime("%Y-%m-%d"):
                    print("\n" + "="*80)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*80)
                    
                    # اجرای همه تحلیل‌ها
                    results = run_all_analyses()
                    
                    # ثبت زمان اجرا
                    last_run_times[current_time] = now.strftime("%Y-%m-%d")
                    
                    print("\n" + "="*80)
                    print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                    print("="*80)
                    
                    # ۶۰ ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(60)
            
            # هر ۳۰ ثانیه یکبار بررسی کن
            time.sleep(30)
            
        except KeyboardInterrupt:
            print("\n" + "="*80)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*80)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)


# ============================================================================
# بخش 4: اجرای اصلی
# ============================================================================

if __name__ == "__main__":
    try:
        print("="*80)
        print("🚀 شروع برنامه تحلیل همبستگی ژنراتور (۲ تحلیل یکپارچه)")
        print("="*80)
        print("📋 لیست تحلیلها:")
        print("   1. تحلیل همبستگی کامل (Long + Wide)")
        print("   2. تحلیل همبستگی (فقط Wide)")
        print("="*80)
        print("⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00")
        print("="*80)
        
        # اجرای زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        import traceback
        traceback.print_exc()
        input("برای خروج Enter بزنید...")